In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.6.1
    Uninstalling fsspec-2024.6.1:
      Successfully uninstalled fsspec-2024.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.6.1 requires fsspec==2024.6.1, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.4/145.4 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.9.0
    Uninstalling fsspec-2024.9.0:
      Successfully uninstalled fsspec-2024.9.0
ERROR: pip's dependency resolver does not currently take into account all the packages that 

In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.data import DataLoader
from torch_geometric.nn import RGCNConv, GATConv, global_mean_pool, LayerNorm
from torch_geometric.datasets import TUDataset
from torch_geometric.transforms import NormalizeFeatures
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.model_selection import KFold
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ENHANCED RGCN MODEL WITH MORE GCNCONV LAYERS AND GAT, LAYER NORM
class EnhancedRGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations=1, dropout=0.5):
        super(EnhancedRGCN, self).__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.norm1 = LayerNorm(hidden_channels)
        self.attn1 = GATConv(hidden_channels, hidden_channels)

        self.conv2 = RGCNConv(hidden_channels, hidden_channels, num_relations)
        self.norm2 = LayerNorm(hidden_channels)
        self.attn2 = GATConv(hidden_channels, hidden_channels)

        self.conv3 = RGCNConv(hidden_channels, hidden_channels, num_relations)
        self.norm3 = LayerNorm(hidden_channels)
        self.attn3 = GATConv(hidden_channels, hidden_channels)

        self.conv4 = RGCNConv(hidden_channels, hidden_channels, num_relations)  # Additional RGCN layer
        self.norm4 = LayerNorm(hidden_channels)  # Additional normalization layer
        self.attn4 = GATConv(hidden_channels, hidden_channels)  # Additional attention layer

        self.conv5 = RGCNConv(hidden_channels, hidden_channels, num_relations)  # Additional RGCN layer
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        batch = data.batch
        edge_type = data.edge_type if hasattr(data, 'edge_type') else torch.zeros(edge_index.size(1), dtype=torch.long, device=edge_index.device)

        # Forward pass with added layers
        x = F.relu(self.norm1(self.conv1(x, edge_index, edge_type)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.attn1(x, edge_index)

        x = F.relu(self.norm2(self.conv2(x, edge_index, edge_type)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.attn2(x, edge_index)

        x = F.relu(self.norm3(self.conv3(x, edge_index, edge_type)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.attn3(x, edge_index)

        x = F.relu(self.norm4(self.conv4(x, edge_index, edge_type)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.attn4(x, edge_index)

        x = F.relu(self.conv5(x, edge_index, edge_type))
        x = global_mean_pool(x, batch)
        return F.log_softmax(x, dim=1)

# TRAIN MODEL FUNCTION WITH REDUCED LR SCHEDULER
def train_model(model, loader, epochs=200, lr=0.001, weight_decay=1e-4):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6)
    criterion = torch.nn.CrossEntropyLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for data in loader:
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, data.y.argmax(dim=1) if data.y.ndim > 1 else data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step(total_loss / len(loader))

    return model

# EVALUATE MODEL FUNCTION
def evaluate_model(model, loader, num_classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data in loader:
            out = model(data)
            preds = out.argmax(dim=1)

            if data.y.ndim == 2:
                labels = data.y.argmax(dim=1)
            else:
                labels = data.y

            all_preds.append(out)
            all_labels.append(labels)

    all_preds = torch.cat(all_preds, dim=0)
    all_labels = torch.cat(all_labels, dim=0).view(-1)

    out_prob = F.softmax(all_preds, dim=1).cpu()

    current_num_classes = out_prob.shape[1]
    if current_num_classes != num_classes:
        if current_num_classes < num_classes:
            padding = num_classes - current_num_classes
            out_prob = torch.cat([out_prob, torch.zeros(out_prob.size(0), padding)], dim=1)
        else:
            out_prob = out_prob[:, :num_classes]

    auc_roc = roc_auc_score(all_labels.cpu().numpy(), out_prob[:, 1], average='weighted') if num_classes == 2 else roc_auc_score(all_labels.cpu().numpy(), out_prob, multi_class='ovr', average='weighted')
    logloss = log_loss(all_labels.cpu().numpy(), out_prob)

    test_acc = (all_preds.argmax(dim=1) == all_labels).sum().item() / len(all_labels)
    f1 = f1_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    precision = precision_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')
    recall = recall_score(all_labels.cpu(), all_preds.argmax(dim=1).cpu(), average='weighted')

    return {
        'accuracy': test_acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'auc_roc': auc_roc,
        'log_loss': logloss
    }

# LOAD AND AUGMENT DATASETS
dataset_nci1 = TUDataset(root='/content/NCI1', name='NCI1', transform=NormalizeFeatures())
dataset_mutag = TUDataset(root='/content/MUTAG', name='MUTAG', transform=NormalizeFeatures())

# SPLIT DATASETS INTO TRAINING AND TESTING SETS (70/30)
train_idx_nci1, test_idx_nci1 = train_test_split(range(len(dataset_nci1)), test_size=0.3, random_state=42)
train_dataset_nci1 = dataset_nci1[train_idx_nci1]
test_dataset_nci1 = dataset_nci1[test_idx_nci1]

train_idx_mutag, test_idx_mutag = train_test_split(range(len(dataset_mutag)), test_size=0.3, random_state=42)
train_dataset_mutag = dataset_mutag[train_idx_mutag]
test_dataset_mutag = dataset_mutag[test_idx_mutag]

train_loader_nci1 = DataLoader(train_dataset_nci1, batch_size=32, shuffle=True)
test_loader_nci1 = DataLoader(test_dataset_nci1, batch_size=32, shuffle=False)

train_loader_mutag = DataLoader(train_dataset_mutag, batch_size=32, shuffle=True)
test_loader_mutag = DataLoader(test_dataset_mutag, batch_size=32, shuffle=False)

# INITIALIZE AND TRAIN MODEL ON NCI1
model_nci1 = EnhancedRGCN(in_channels=dataset_nci1.num_node_features, hidden_channels=64, out_channels=dataset_nci1.num_classes, dropout=0.5)
train_model(model_nci1, train_loader_nci1, epochs=200)

# EVALUATE ON NCI1 DATASET
print("\nResults on NCI1 Dataset Before Transfer Learning:")
nci1_results = evaluate_model(model_nci1, test_loader_nci1, num_classes=dataset_nci1.num_classes)
print(nci1_results)

# TRANSFER LEARNING TO MUTAG DATASET
model_mutag = EnhancedRGCN(in_channels=dataset_mutag.num_node_features, hidden_channels=64, out_channels=dataset_mutag.num_classes, dropout=0.5)
state_dict_nci1 = model_nci1.state_dict()
state_dict_mutag = model_mutag.state_dict()

for key in state_dict_nci1:
    if key in state_dict_mutag and state_dict_nci1[key].shape == state_dict_mutag[key].shape:
        state_dict_mutag[key] = state_dict_nci1[key]

model_mutag.load_state_dict(state_dict_mutag, strict=False)
train_model(model_mutag, train_loader_mutag, epochs=200)

# EVALUATE ON MUTAG DATASET
print("\nResults on MUTAG Dataset After Transfer Learning:")
mutag_results = evaluate_model(model_mutag, test_loader_mutag, num_classes=dataset_mutag.num_classes)
print(mutag_results)



Processing...
Done!
/usr/local/lib/python3.10/dist-packages/torch_geometric/io/fs.py:215: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f, map_location)
Pr


Results on NCI1 Dataset Before Transfer Learning:


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2922: UserWarning: The y_pred values do not sum to one. Starting from 1.5 thiswill result in an error.
  warnings.warn(


{'accuracy': 0.7461476074614761, 'f1': 0.7457495140721258, 'precision': 0.7464250956409516, 'recall': 0.7461476074614761, 'auc_roc': 0.8024293374629963, 'log_loss': 0.5600796565682817}

Results on MUTAG Dataset After Transfer Learning:
{'accuracy': 0.8070175438596491, 'f1': 0.8105802968960863, 'precision': 0.8187134502923976, 'recall': 0.8070175438596491, 'auc_roc': 0.8917378917378918, 'log_loss': 0.4418088477505271}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2922: UserWarning: The y_pred values do not sum to one. Starting from 1.5 thiswill result in an error.
  warnings.warn(
